# 02 · 静态扫描件切片、人工复核与模板引导单线笔顺轨迹恢复

本 Notebook 完成离线扫描件到有向有序单线矢量轨迹（`0..1000` 坐标系）的转换：
1. **扫描矫正与切片（`hwbuilder.segment`）**：四角 ArUco 透视校正、光照归一化、青/红色格线滤除、Sauvola/Otsu 混合二值化与连通域去噪；
2. **人工复核（`review.html`）**：一键导出交互式 HTML 复核页，点击剔除错切/污渍切片并下载 `kept_ids.json`；
3. **笔顺轨迹恢复（`hwbuilder.skeleton_align`）**：Zhang-Suen 细化 + 距离变换毛刺裁剪 + 标准笔顺模板（MakeMeAHanzi / Hershey）TPS 非刚性配准，输出 `trajectories.json` 供 Notebook 03 微调与导出。

In [ ]:
import json
import os
import sys
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip -q install opencv-contrib-python-headless pymupdf pillow numpy pydantic

REPO_ROOT = Path("/content/handwrite") if IN_COLAB and Path("/content/handwrite").exists() else Path.cwd().parents[1]
for sub in ("packages/hwfont_sdk", "colab_builder", "backend"):
    p = str(REPO_ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

WRITER_NAME = "zhangsan"
WORK_ROOT = Path("/content/drive/MyDrive/handwrite_builder") / WRITER_NAME if IN_COLAB else Path("./work") / WRITER_NAME
MANIFEST_PATH = WORK_ROOT / "01_sheets" / "manifest.json"
SCANS_DIR = WORK_ROOT / "02_scans"          # 请将 page_000.png / page_001.png ... 放入此处
CROPS_DIR = WORK_ROOT / "02_crops"
TRAJ_PATH = WORK_ROOT / "02_trajectories.json"
print("WORK_ROOT:", WORK_ROOT)

## 1. 批量透视对齐、去网格与字形切片

将 `02_scans/` 中按页码排序的扫描图片逐页对齐到 `manifest.json`，并导出交互式复核页面 `review.html`。

In [ ]:
from hwbuilder.sheet_gen import SheetManifest
from hwbuilder.segment import SegmentConfig, segment_scan, export_review_html

manifest = SheetManifest.load(MANIFEST_PATH)
seg_cfg = SegmentConfig(
    target_dpi=300,
    grid_color=manifest.grid_color,
    binarize_method="hybrid",
)

scan_files = sorted(
    p for p in SCANS_DIR.iterdir()
    if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
) if SCANS_DIR.exists() else []

all_crops = []
for page_idx, scan_path in enumerate(scan_files):
    if page_idx >= len(manifest.pages):
        break
    crops, diag = segment_scan(
        scan_path,
        manifest,
        page_index=page_idx,
        config=seg_cfg,
        save_crops_dir=CROPS_DIR,
    )
    all_crops.extend(crops)
    print(f"Page {page_idx}: align={diag['align_method']}, kept={diag['kept_initial']}/{diag['total_cells']}, flagged={diag['flagged_cells']}")

review_html = export_review_html(all_crops, WORK_ROOT / "review.html")
print("交互式复核页面已生成 ->", review_html)

## 2. （可选）载入人工复核结果 `kept_ids.json`

下载并打开上一步生成的 `review.html`，点击剔除空白、串格或误切切片，点击右上角 **导出 kept_ids.json** 并上传到 `WORK_ROOT / "kept_ids.json"`。如未提供该文件，则自动保留所有未标记 `empty` 的切片。

In [ ]:
KEPT_JSON = WORK_ROOT / "kept_ids.json"
if KEPT_JSON.exists():
    kept_ids = set(json.loads(KEPT_JSON.read_text(encoding="utf-8")).get("kept_ids", []))
    filtered_crops = [c for c in all_crops if c.cell_id in kept_ids]
    print(f"已应用 kept_ids.json：保留 {len(filtered_crops)} / {len(all_crops)} 个切片")
else:
    filtered_crops = [c for c in all_crops if c.kept]
    print(f"未检测到 kept_ids.json，自动保留非空切片：{len(filtered_crops)} / {len(all_crops)}")

## 3. 骨架化 + 标准笔顺先验 TPS 非刚性配准

如已准备 MakeMeAHanzi 的 `graphics.txt`（含 9000+ 汉字标准中线笔顺，遵循其开源协议），可在下方设置 `MAKEMEAHANZI_PATH` 以获得高精度汉字笔顺恢复；拉丁字母/数字/数学符号将自动使用内置 Hershey 单线先验。

In [ ]:
from hwbuilder.skeleton_align import (
    VectorizeConfig,
    load_builtin_priors,
    load_makemeahanzi_priors,
    vectorize_glyph,
)

priors = load_builtin_priors()
MAKEMEAHANZI_PATH = WORK_ROOT.parent / "priors" / "graphics.txt"
if MAKEMEAHANZI_PATH.exists():
    priors.update(load_makemeahanzi_priors(MAKEMEAHANZI_PATH))
    print(f"已加载 MakeMeAHanzi + 内置先验共 {len(priors)} 字")
else:
    print(f"未找到 graphics.txt，当前使用内置先验 {len(priors)} 字（其余字形自动启用最小转角启发式追踪）")

vec_cfg = VectorizeConfig()
records = []
method_counts = {"tps_template": 0, "heuristic_trace": 0, "empty": 0}

for crop in filtered_crops:
    res = vectorize_glyph(crop.binary, char=crop.char, priors=priors, config=vec_cfg)
    method_counts[res.method] = method_counts.get(res.method, 0) + 1
    if not res.strokes:
        continue
    records.append({
        "cell_id": crop.cell_id,
        "char": crop.char,
        "method": res.method,
        "quality": res.quality,
        "stroke_width_px": res.estimated_stroke_width_px,
        "strokes": [s.tolist() for s in res.strokes],
    })

TRAJ_PATH.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")
print("轨迹恢复统计:", method_counts)
print(f"已保存 {len(records)} 条有向单线轨迹 -> {TRAJ_PATH}")